In [1]:
import sys
import os
import json
import urllib3
import requests
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [7]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-04-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
summary = 'alynforensics.com'

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastObserved >= "{start}" and (indicatorActive EQ true OR indicatorActive EQ false)'

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{summary}?tql={tql}&fields=threatAssess,associatedGroups,observations&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    observed_src = pd.json_normalize([result['data'] for result in final_results if 'data' in result])
    observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
    print(f"\nRetrieved {len(observed_src)} observed indicators.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 3 observed indicators.


In [8]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,lastObserved,privateFlag,active,activeLocked,hostName,dnsActive,whoisActive,legacyLink,associatedGroups.data,indicator
0,6755399443022985,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T13:46:43Z,5.0,98,2.5,...,2025-04-24T00:00:00Z,False,True,False,alynforensics.com,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 6755399443003054, 'dateAdded': '2025-0...",alynforensics.com
1,6755399443022985,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T13:46:43Z,5.0,98,2.5,...,2025-04-24T00:00:00Z,False,True,False,alynforensics.com,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 6755399443003054, 'dateAdded': '2025-0...",alynforensics.com
2,6755399443022985,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T13:46:43Z,5.0,98,2.5,...,2025-04-24T00:00:00Z,False,True,False,alynforensics.com,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 6755399443003054, 'dateAdded': '2025-0...",alynforensics.com


In [5]:
final_results

[{'data': [{'id': 5288144,
    'dateAdded': '2025-02-04T19:30:40Z',
    'ownerId': 9,
    'ownerName': 'HTOC Org',
    'webLink': 'https://hvs.threatconnect.com/#/details/indicators/5288144',
    'type': 'Address',
    'lastModified': '2025-04-24T15:10:00Z',
    'rating': 4.0,
    'confidence': 89,
    'threatAssessRating': 4.0,
    'threatAssessConfidence': 89.0,
    'threatAssessScore': 587,
    'threatAssessScoreObserved': 167,
    'threatAssessScoreFalsePositive': 0,
    'source': 'Mandiant Threat Intelligence',
    'description': 'UNC5837 Related indicators reported by Mandiant',
    'summary': '65.21.203.39',
    'observations': 96216,
    'lastObserved': '2025-04-12T00:00:00Z',
    'privateFlag': False,
    'active': True,
    'activeLocked': False,
    'associatedGroups': {'data': [{'id': 5629499536006730,
       'dateAdded': '2025-03-31T14:44:58Z',
       'ownerId': 9,
       'ownerName': 'HTOC Org',
       'webLink': 'https://hvs.threatconnect.com/#/details/groups/56294995360

OPTIONS request failed with status code: 401
Response: {"message":"ERROR (Unauthorized): You are not authorized to perform the requested operation.","status":"Error"}
